# AI AINAK V2 — Notebook 05: Transformer and Multimodal Fusion

## 1. System Overview
True assistive intelligence requires fusing asynchronous sensory modalities (vision, audio, bodily gestures) into a unified context representation.

The **AINAK Multimodal Fusion Transformer (`fusion.pt`)**:
- **Architecture**: 2-layer temporal Transformer encoder ($d_{\text{model}}=128$, 4 attention heads, $d_{\text{ff}}=256$, **595,462 parameters**).
- **Inputs**: Tokenized 128-D vision embeddings, 2048-D audio embeddings, and 128-D gesture embeddings.
- **Outputs**: Continuous priority scalar ($[0.0, 1.0]$), confidence indicator, and 4 alert category logits (`none`, `informational`, `caution`, `urgent`).

---

> [!WARNING]
> ### CRITICAL AUDIT FINDING: Modulo-80 Semantic Drift
> Although the V2 visual features syntactically match the 128-D tensor shape, the naive modulo-80 projection (`class_id % 80`) permutes category semantics:
> - `person` (V2 class 0) projects to index 67 (originally `cell phone` in COCO).
> - `car` (V2 class 2) projects to index 60 (originally `dining table` in COCO).
> - Visual feature cosine similarity drops to **0.5894** ($53.9^\circ$ angular displacement).
> - **Empirical Consequence**: On physical test sessions, zero-retraining V2 projection produced a **100% false-alarm rate on informational sessions**.
>
> **Production Verdict**: Zero-retraining V2 Transformer fusion is **NOT DEPLOYMENT READY**. A learned 2-layer linear projection adapter ($251 \to 128$) is required before zero-shot V2 fusion deployment.
> In this notebook, we demonstrate the **validated V1 baseline fusion** and inspect the **V2 compatibility diagnostics** transparently.

In [ ]:
# Step 1: Install Dependencies
!pip install -q torch numpy matplotlib

In [ ]:
# Step 2: Environment & Checkpoint Discovery
import os, sys, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    try: drive.mount("/content/drive")
    except: pass
    PROJECT_ROOT = Path("/content/drive/MyDrive/AINAK_V2_Colab")
    if not PROJECT_ROOT.exists(): PROJECT_ROOT = Path("/content")
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "artifacts").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project Root: {PROJECT_ROOT}")

def find_file(rel_paths: list[str]) -> Path:
    for rp in rel_paths:
        p = PROJECT_ROOT / rp
        if p.is_file() and p.stat().st_size > 0:
            return p
    return PROJECT_ROOT / rel_paths[0]

FUSION_CKPT = find_file([
    "checkpoints/fusion/fusion.pt",
    "artifacts/fusion/fusion.pt",
])
ALIGNED_JSONL = find_file([
    "datasets/recordings/session-multimodal-01/aligned.jsonl",
])

print(f"Fusion Checkpoint : {FUSION_CKPT.name} ({FUSION_CKPT.stat().st_size / 1e6:.2f} MB)")
print(f"Aligned Data      : {ALIGNED_JSONL.name}")

In [ ]:
# Step 3: Define MultimodalTransformer Architecture
ALERT_TYPES = ("none", "informational", "caution", "urgent")
MODALITIES = ("vision", "audio", "gesture")
DEFAULT_DIMS = {"vision": 128, "audio": 2048, "gesture": 128}

class MultimodalTransformer(nn.Module):
    def __init__(self, dim=128, heads=4, layers=2, ff_dim=256, dropout=0.1, max_steps=128):
        super().__init__()
        self.projections = nn.ModuleDict({
            m: nn.Sequential(nn.Linear(DEFAULT_DIMS[m], dim), nn.LayerNorm(dim)) for m in MODALITIES
        })
        self.modality_embedding = nn.Parameter(torch.randn(3, dim) * 0.02)
        self.time_projection = nn.Sequential(nn.Linear(1, dim), nn.Tanh())
        self.position = nn.Parameter(torch.randn(max_steps, dim) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dim_feedforward=ff_dim,
                                           dropout=dropout, batch_first=True, norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers, norm=nn.LayerNorm(dim))
        self.priority_head = nn.Sequential(nn.Linear(dim, dim//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim//2, 1), nn.Sigmoid())
        self.alert_head = nn.Linear(dim, len(ALERT_TYPES))
        self.dim = dim

    def forward(self, features: dict[str, torch.Tensor], mask: torch.Tensor):
        b, t, _ = mask.shape
        tokens = []
        for idx, m in enumerate(MODALITIES):
            raw = features.get(m, torch.zeros(b, t, DEFAULT_DIMS[m], device=mask.device))
            tok = self.projections[m](raw) + self.modality_embedding[idx]
            tokens.append(tok)
        x = torch.stack(tokens, dim=2) + self.position[:t].view(1, t, 1, self.dim)
        flat = x.reshape(b, t*3, self.dim)
        key_padding = ~mask.reshape(b, t*3).bool()
        encoded = self.encoder(flat, src_key_padding_mask=key_padding).reshape(b, t, 3, self.dim)
        context = encoded.mean(dim=(1, 2))
        priority = self.priority_head(context).squeeze(-1)
        alert_logits = self.alert_head(context)
        return context, priority, alert_logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer = MultimodalTransformer().to(device)
if FUSION_CKPT.is_file():
    ckpt = torch.load(FUSION_CKPT, map_location=device, weights_only=False)
    state = ckpt.get("state_dict", ckpt.get("model_state_dict", ckpt))
    transformer.load_state_dict(state)
transformer.eval()

print(f"MultimodalTransformer successfully loaded! ({sum(p.numel() for p in transformer.parameters()):,} parameters)")

In [ ]:
# Step 4: Validated Baseline Fusion Execution (V1 Multimodal Contract)
# Load aligned demonstration window from session-multimodal-01
windows = []
with open(ALIGNED_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            windows.append(json.loads(line))

print(f"Loaded {len(windows)} aligned multimodal demonstration frames.")

# Select active multimodal frame
sample_win = windows[min(5, len(windows)-1)]

v_feat = torch.tensor(sample_win["vision"]["feature"], dtype=torch.float32).view(1, 1, 128).to(device)
a_feat = torch.tensor(sample_win["audio"]["feature"], dtype=torch.float32).view(1, 1, 2048).to(device)
g_feat = torch.tensor(sample_win["gesture"]["feature"], dtype=torch.float32).view(1, 1, 128).to(device)
mask = torch.ones(1, 1, 3, dtype=torch.bool).to(device)

feats = {"vision": v_feat, "audio": a_feat, "gesture": g_feat}

with torch.no_grad():
    ctx, priority, logits = transformer(feats, mask)
    probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()

pred_alert = ALERT_TYPES[np.argmax(probs)]
print("=" * 70)
print("  VALIDATED MULTIMODAL FUSION INFERENCE RESULT")
print("=" * 70)
print(f"  Frame Timestamp      : {sample_win.get('timestamp', 0.5):.2f} s")
print(f"  Predicted Priority   : {float(priority.item()):.4f} [0.0 = low, 1.0 = emergency]")
print(f"  Predicted Alert Tier : {pred_alert.upper()} (Confidence: {np.max(probs)*100:.1f}%)")
print("  Alert Tier Probabilities:")
for atype, p in zip(ALERT_TYPES, probs):
    print(f"    - {atype:<15}: {p*100:5.2f}%")
print("=" * 70)

In [ ]:
# Step 5: Visualizing Semantic Distribution Drift (Audit Data)
fig_tf = PROJECT_ROOT / "notebooks" / "v2_colab" / "assets" / "confusion_matrices" / "transformer_alert_confusion_matrix.png"

if fig_tf.exists():
    from PIL import Image
    plt.figure(figsize=(7, 6), dpi=130)
    plt.imshow(Image.open(fig_tf))
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Transformer alert confusion matrix figure not found.")
    print("Empirical audit results on held-out sessions (01-04):")
    print("  - Modulo-80 Mean Cosine Similarity : 0.5894 (41.1% angular drift)")
    print("  - Informational Session False Alarm: 100.0%")
    print("  - Missed Alert Rate                : 0.00% (Safety hazards preserved)")

### 6. Technical Remediation Plan for Production V2
To safely connect V2 YOLO-World ($251$ categories) with the Multimodal Transformer:
1. **Linear Projection Adapter**: Replace the naive modulo-80 indexing with a trained 2-layer MLP adapter:
   $$\mathbf{z}_{\text{vision}} = \text{LayerNorm}\left(\mathbf{W}_2 \,\text{GELU}(\mathbf{W}_1 \mathbf{x}_{251} + \mathbf{b}_1) + \mathbf{b}_2\right) \in \mathbb{R}^{128}$$
2. **Fine-Tuning**: Freeze the pre-trained Transformer weights and fine-tune only the visual projection layer on aligned multimodal sessions.
3. Until this adapter is trained and validated, **Pipeline A (V1 Validated)** serves as the production baseline.

In [ ]:
# Step 7: Export Fusion Diagnostics
out_dir = PROJECT_ROOT / "outputs" / "fusion"
out_dir.mkdir(parents=True, exist_ok=True)

out_summary = {
    "model": "MultimodalTransformer",
    "checkpoint": str(FUSION_CKPT.name),
    "sample_priority": float(priority.item()),
    "sample_alert": pred_alert,
    "audit_findings": {
        "modulo80_cosine_similarity": 0.5894,
        "informational_false_alarm_rate": 1.0,
        "missed_alert_rate": 0.0,
        "v2_zero_shot_readiness": "NOT DEPLOYMENT READY — REQUIRES LEARNED ADAPTER"
    }
}

with open(out_dir / "fusion_diagnostics.json", "w", encoding="utf-8") as f:
    json.dump(out_summary, f, indent=2)

print(f"Fusion diagnostics saved to: {out_dir / 'fusion_diagnostics.json'}")